In [1]:
adapter_path = "paul-stansifer/qw3-gemma2-9b-1x6e-4" #input("Huggingface model path:")
if adapter_path.count("/") != 1:
    raise Exception(f"'{adapter_path}' doesn't seem correct!")

In [2]:
%%capture
%pip install peft torch tqdm transformers bitsandbytes


In [3]:
import transformers
transformers.utils.import_utils._bitsandbytes_available = transformers.utils.import_utils._is_package_available("bitsandbytes")
# print(transformers.utils.import_utils._is_package_available("bitsandbytes"))
# print(transformers.utils.import_utils.is_torch_available())
# print(transformers.utils.import_utils.is_bitsandbytes_available())
# print(transformers.utils.import_utils._is_package_available("bitsandbytes", True))
# print(transformers.utils.import_utils._bitsandbytes_available)

/home/paul/src/qwantzle-search/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

qwantz = ["q8_0", "q4_k_m", "q2_k"]  # I must've mispelled something...

In [5]:
import json
from urllib.request import urlopen, Request
from urllib.error import URLError, HTTPError

url = f"https://huggingface.co/{adapter_path}/raw/main/adapter_config.json"
req = Request(url, headers={"User-Agent": "python-urllib"})

with urlopen(req) as resp:
    cfg = json.load(resp)
base_model_id = cfg["base_model_name_or_path"]
print("base model identified:", base_model_id)


base model identified: unsloth/gemma-2-9b-bnb-4bit


In [6]:
short_name = adapter_path.split("/")[-1]

In [7]:
# Load base model on CPU (force CPU to avoid GPU OOM).
# Use torch_dtype=torch.float32 or float16 depending on base availability.
#base = AutoModelForCausalLM.from_pretrained(base_model_id, device_map="auto", low_cpu_mem_usage=True)
base = AutoModelForCausalLM.from_pretrained(base_model_id, device_map="cpu", low_cpu_mem_usage=True)

In [ ]:
# Wrap with the adapter
model_with_adapter = PeftModel.from_pretrained(base, adapter_path, device_map="cpu")

/home/paul/src/qwantzle-search/.venv/lib/python3.12/site-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'peft_version'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


: 

In [ ]:
merged_model = model_with_adapter.merge_and_unload()

In [ ]:


merged_model.save_pretrained("/tmp/merged-model", safe_serialization=True)

# Save tokenizer (not sure if this is needed)
tokenizer = AutoTokenizer.from_pretrained(base_model_id, use_fast=False)
tokenizer.save_pretrained("/tmp/merged-model")

print("Merged model saved to", "/tmp/merged-model")

/home/paul/src/qwantzle-search/.venv/lib/python3.12/site-packages/peft/tuners/lora/bnb.py:348: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [ ]:
%%bash
python /home/paul/others/llama.cpp/convert_hf_to_gguf.py \
  /tmp/merged-model \
  --outfile /tmp/merged-model-f16.gguf \
  --outtype f16

INFO:hf-to-gguf:Loading model: merged-model
INFO:hf-to-gguf:Model architecture: SmolLM3ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00003.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00003.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00003.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,           torch.float32 --> F16, shape = {2048, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float32 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float32 --> F16, shape = {11008, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float32 --> F16, shape = {2048, 11008}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float32 --> F16, shape = {2048, 11008}
INFO:hf-to-gguf:blk.0.ffn_norm.weight, 

In [ ]:
%%bash
/home/paul/others/llama.cpp/build/bin/llama-quantize \
    /tmp/merged-model-f16.gguf /tmp/merged-model-q8_0.gguf Q8_0 --log-disable

main: build = 8076 (d61290111)
main: built with GNU 13.3.0 for Linux x86_64
main: quantizing '/tmp/merged-model-f16.gguf' to '/tmp/merged-model-q8_0.gguf' as Q8_0
llama_model_loader: loaded meta data with 23 key-value pairs and 326 tensors from /tmp/merged-model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = smollm3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged Model
llama_model_loader: - kv   3:                         general.size_label str              = 3.1B
llama_model_loader: - kv   4:                        smollm3.block_count u32              = 36
llama_model_loader: - kv   5:                     smollm3.context_length u32              = 65536
llama_mode


main: quantize time =  9286.54 ms
main:    total time =  9286.54 ms


In [ ]:
%%bash
/home/paul/others/llama.cpp/build/bin/llama-quantize \
    /tmp/merged-model-f16.gguf /tmp/merged-model-q4_k_m.gguf Q4_K_M --log-disable

main: build = 8076 (d61290111)
main: built with GNU 13.3.0 for Linux x86_64
main: quantizing '/tmp/merged-model-f16.gguf' to '/tmp/merged-model-q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 23 key-value pairs and 326 tensors from /tmp/merged-model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = smollm3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged Model
llama_model_loader: - kv   3:                         general.size_label str              = 3.1B
llama_model_loader: - kv   4:                        smollm3.block_count u32              = 36
llama_model_loader: - kv   5:                     smollm3.context_length u32              = 65536
llama_


main: quantize time = 28728.33 ms
main:    total time = 28728.33 ms


In [ ]:
%%bash
#/home/paul/others/llama.cpp/build/bin/llama-quantize --help
/home/paul/others/llama.cpp/build/bin/llama-quantize \
    /tmp/merged-model-f16.gguf /tmp/merged-model-q2_k.gguf Q2_K --log-disable

main: build = 8076 (d61290111)
main: built with GNU 13.3.0 for Linux x86_64
main: quantizing '/tmp/merged-model-f16.gguf' to '/tmp/merged-model-q2_k.gguf' as Q2_K
llama_model_loader: loaded meta data with 23 key-value pairs and 326 tensors from /tmp/merged-model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = smollm3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged Model
llama_model_loader: - kv   3:                         general.size_label str              = 3.1B
llama_model_loader: - kv   4:                        smollm3.block_count u32              = 36
llama_model_loader: - kv   5:                     smollm3.context_length u32              = 65536
llama_mode


main: quantize time = 17191.15 ms
main:    total time = 17191.15 ms


In [ ]:
from huggingface_hub import upload_file

for q in qwantz:
    upload_file(
        path_or_fileobj=f"/tmp/merged-model-{q}.gguf",
        path_in_repo=f"{short_name}-{q}.gguf",
        repo_id=adapter_path,
        repo_type="model"
    )

print(f"Files uploaded to https://huggingface.co/{adapter_path}")

Processing Files (1 / 1): 100%|██████████| 3.28GB / 3.28GB, 6.01MB/s  
New Data Upload: 100%|██████████| 2.99GB / 2.99GB, 6.01MB/s  
Processing Files (1 / 1): 100%|██████████| 1.92GB / 1.92GB, 5.59MB/s  
New Data Upload: 100%|██████████| 1.73GB / 1.73GB, 5.59MB/s  
Processing Files (1 / 1): 100%|██████████| 1.25GB / 1.25GB, 5.31MB/s  
New Data Upload: 100%|██████████| 1.02GB / 1.02GB, 5.31MB/s  


Files uploaded to https://huggingface.co/paul-stansifer/qw3-smollm3-3b-1x2e-4
